# C6 — OOF Statistics, Calibration, and XAI

**Protocol:** `v1.0.0`  
**Protocol hash:** `d42337690181f1054297f514934ad0c98bb718223bc06d8de5569f40a184ee32`

Primary evidence berasal dari pooled OOF predictions pada 5-fold patient-level CV. Bootstrap bersifat conditional on fitted CV models dan tidak melakukan retraining pada setiap replicate. Notebook ini tidak membuka official NIH test.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'run_experiment.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.reporting.canonical import load_canonical_context

context = load_canonical_context(PROJECT_ROOT)
protocol_dir = context.protocol_dir
statistics_dir = protocol_dir / 'statistics'
xai_dir = protocol_dir / 'xai'
assert (statistics_dir / '_SUCCESS').is_file(), 'C6 belum lengkap'
assert (xai_dir / '_SUCCESS').is_file(), 'XAI C6 belum lengkap'
print('protocol_hash:', context.protocol_hash)

## Metrik pooled OOF dan stabilitas antar-fold

In [ ]:
metrics = pd.read_csv(statistics_dir / 'metrics.csv')
metrics[['model', 'roc_auc_pooled', 'roc_auc_mean', 'roc_auc_sd', 'ap_pooled', 'brier_score', 'sensitivity_0.5', 'specificity_0.5']]

## Paired patient-cluster bootstrap (2.000 replicate)

In [ ]:
bootstrap = json.loads((statistics_dir / 'bootstrap_comparisons.json').read_text(encoding='utf-8'))
bootstrap_table = pd.DataFrame([{'comparison': name, **values} for name, values in bootstrap.items()])
bootstrap_table[['comparison', 'model_a', 'model_b', 'delta_auc', 'ci_low', 'ci_high', 'n_boot']]

CI yang melintasi nol ditulis sebagai **bukti OOF belum memisahkan model secara jelas**, bukan sebagai bukti equivalence.

## Diagnostic calibration

In [ ]:
display(Image(filename=str(statistics_dir / 'calibration' / 'reliability_diagram.png')))

Raw sigmoid diperlakukan sebagai **skor model**, bukan calibrated probability, karena model dilatih dengan weighted BCE.

## Fold-specific image-conditioned SHAP

In [ ]:
shap_importance = pd.read_csv(xai_dir / 'shap' / 'mean_absolute_shap.csv')
display(shap_importance)
display(Image(filename=str(xai_dir / 'shap' / 'summary.png')))

Mean absolute SHAP adalah rata-rata magnitudo atribusi kondisional pada 200 kasus OOF; setiap atribusi dikondisikan pada X-ray aktual yang berbeda dan bukan joint cross-modal attribution universal.

## Paired OOF Grad-CAM S2 vs S3

In [ ]:
cases = pd.read_csv(xai_dir / 'gradcam' / 'case_artifacts.csv')
display(cases[['category', 'image_index', 'fold', 'true_label', 's2_probability', 's3_probability']])
display(Image(filename=str(xai_dir / 'gradcam' / 'paired_gradcam_grid.png')))